<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>



# Lane Control

Now that we have an estimate of our lateral displacement $d$ and our angle relative to the lane $\phi$, our task here is to decide how to control the wheels so that the robot drives straight down the lane. Even if we place the robot directly in the middle of the lane, simply commanding it to go straight (equal forward speed on each wheel) will obviously not work because: 1) there are disturbances and the robot will deviate (we mitigate this as much as possible by the process of [wheel calibration](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-motor-odometry-calibration.html) but it will never be perfect) and 2) there are turns in the road that we need to adapt to.

We will need to use our information about $d$ and $\phi$ to set the speeds of the wheels, which is feedback control.

## Inverse Kinematics

One first note is that it will be intuitively simpler to build our controller over the linear speed ($v$) and the angular speed ($\omega$) of the Duckiebot since we can easily convert these to right and left wheel speeds using the _inverse kinematics_ equations. For more details see the [kinematics and odometry LX](https://github.com/duckietown/lx-kinematics-odometry).

## "PID-Inspired" Control

In the [Duckietown control LX](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-pid-control.html) we have learned about a very simple feedback control strategy called "PID", where the control is a linear combination of the proportial error, the change in error (derivative) and the accumulated error (integral). The was simple to implement when our state was one dimensional, and our control was one-dimensional. In this case we have two states ($d$ and $phi$) and two control outputs ($v$ and $\omega$). We can reduce the control outputs to one dimension by simply assuming that we want the robot to travel at a fixed velocity $v=\bar{v}$ and only worry about how to control the steering of the robot through $\omega$.

We subsequently propose the following control law:

$$
u_t = K^d_d d_t + K^d_{\phi} \phi_t + K^I_d d_t,
$$

where:

* $K^d_d$ is the proportional gain weighting the diplacement error.
* $K^d_\phi$ is the proportional gain weighting the angular error.
* $K^I_d$ is the integral gain weighting the accumulated diplacement error.

We should note that, although this closely resembles a PID controller, it technically is not one. In fact this controller can be derived from a "pole-placement" style of approach which is beyong the scope of this notebook but for details please refer to [these slides](https://docs.google.com/presentation/d/15awvCABgWQhjMkF6bMfAIevz-468u92V/edit?slide=id.p29#slide=id.p29).

## The controller in `dt-ros2-core`

The controller that runs on the robot is the `lane_controller_node` of the [lane_control package](../packages/dt-ros2-core/packages/lane_control/) ([`lane_control/lane_controller_node.py`](../packages/dt-ros2-core/packages/lane_control/lane_control/lane_controller_node.py)). It subscribes to the estimate of the lane filter (`lane_filter_node/lane_pose`) and, every time a new estimate arrives, publishes a `duckietown_msgs/Twist2DStamped` command $(v, \omega)$ on `lane_controller_node/car_cmd`. The ROS 2 port implements the two proportional terms of the law above (the integral term is not implemented yet in `dt-ros2-core`; adding it is a nice exercise), with the gains written as positive numbers and the sign in the law, and with saturations on the errors:

$$
\begin{aligned}
e_d &= \mathrm{clip}(d - d_{\text{offset}},\ -d_{\text{thres}},\ d_{\text{thres}}) \\
e_\phi &= \mathrm{clip}(\phi,\ \theta_{\text{thres,min}},\ \theta_{\text{thres,max}}) \\
v &= \bar{v} \\
\omega &= -k_d\, e_d - k_\theta\, e_\phi + \omega_{ff}
\end{aligned}
$$

A positive lateral deviation (too far left) or a positive heading error (pointing left) must produce a negative angular velocity (turn right), hence the minus signs.

## Tuning the Gains

Tuning the controller gains is one of the most important aspects for the stability and performance of your Duckiebot.

We can largely follow the strategy introduced in the [control learning experience](https://github.com/duckietown/lx-control), if we consider that our reference is the middle of the road ($d=0$) and that, for relatively small values of $\phi$, the $\phi$ error acts similarly to a derivative term for tracking $d=0$. Put another way, we can intuitively understand that if the robot is not directly pointing down the lane ($\phi \neq 0$) then this will induce a change in the lateral displacement error, which is very similar to what the derivative term does in a PID controller. So, following the PID intuition we should expect that:

* The $k_d$ gain affects the magnitude of corrections. Too high a value leads to oscillations, while too low a value results in sluggish response (similar to the proportonal gain in a PID controller);
* The $k_\theta$ gain smooths out the response by reducing overshoot but can amplify noise (similar to the derivative gain in a PID controller);
* An integral gain on $d$ (if you add one) reduces steady-state errors but can introduce instability if over-tuned (similar to the integral gain in a PID controller).

The parameters are declared by the node, with the defaults also listed in the config of the [lane controller](../packages/dt-ros2-core/packages/lane_control/config/lane_controller.yaml). Here are the defaults:


```yaml
lane_controller:
  ros__parameters:
    v_bar: 0.25            # nominal forward velocity (m/s)
    k_d: 3.0               # proportional gain for the lateral deviation d
    k_theta: 8.0           # proportional gain for the heading error phi
    d_thres: 0.3           # the lateral error is clipped to [-d_thres, d_thres] (m)
    theta_thres_min: -0.6  # the heading error is clipped to [theta_thres_min, theta_thres_max] (rad)
    theta_thres_max: 0.6
    d_offset: 0.0          # goal lateral offset from the lane center (m)
    omega_ff: 0.0          # feed-forward angular velocity (rad/s)
```

The are defined as follows:

- `v_bar`:  Nominal forward velocity (m/s). Note that this is also dependant on the `gain` that is tuned during [wheel calibration](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-motor-odometry-calibration.html).

- `k_d`: Proportional gain on lateral error $d$ (applied with a minus sign, see above). Larger magnitude = more aggressive lateral correction.

- `k_theta`:  Proportional gain on heading error $\phi$. Typically larger than `k_d` since heading error is easier to correct quickly.

- `d_thres`, `theta_thres_min`, `theta_thres_max`: saturation of the two errors, so that a wild estimate (e.g. right after the robot is placed in the lane) cannot produce a wild command.

- `d_offset`: lets you drive off-center on purpose (e.g. to avoid an obstacle on one side).

- `omega_ff`: a constant feed-forward turning rate, useful to compensate a systematic bias.

The node reads its parameters every time it computes a command, so you can tune them live on the running demo, either from a terminal inside the code editor:

```bash
    ros2 param set /ROBOT_NAME/lane_controller_node k_d 2.5
    ros2 param set /ROBOT_NAME/lane_controller_node v_bar 0.2
```

or with the sliders of the cell below (the demo must be running on your robot). The values go back to the defaults when you restart the demo: once you are happy with them, write them in the YAML file (or pass them in the launch file, see [notebook 06](./06_building_the_demo.ipynb)).


In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ParamPanel, TopicPlot

ROBOT_NAME = "ROBOT_NAME"   # <-- the name of your (real or virtual) Duckiebot
ros = RosSession(ROBOT_NAME)

%matplotlib widget
# sliders for the controller gains: "Read from robot" fetches the current values, "Send" applies yours
display(ParamPanel(ros, "lane_controller_node", {
    "v_bar": (0.0, 0.5, 0.01),
    "k_d": (0.0, 10.0, 0.1),
    "k_theta": (0.0, 15.0, 0.1),
    "d_offset": (-0.1, 0.1, 0.005),
    "omega_ff": (-2.0, 2.0, 0.05),
}))
# the estimate the controller is acting on
TopicPlot(ros, "lane_filter_node/lane_pose", "duckietown_msgs/msg/LanePose", ["d", "phi"])


For more details about PID control please refer to the [Control Learning Experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-pid-control.html).

The velocity and steering values are turned into actuator values using inverse kinematics by [the kinematics node](../packages/dt-ros2-core/packages/robots/duckiebot/dagu_car/dagu_car/kinematics_node.py), which runs on the robot as part of the `ros2-core/default` stack: it subscribes to `car_cmd_switch_node/cmd` and publishes `wheels_driver_node/wheels_cmd`. For more details about direct and inverse kinematics refer to the [Kinematics and Odometry Learning Experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-modeling-kinematics.html).

We now have understood the entire autonomy stack, from the data that comes in through the sensors (camera and encoders) to the actuator commands that are sent to the wheels. One last piece to discuss is how we build a "finite state machine" that sits on top of this stack and manages what the macro-level behaviour of the robot should be.

In this relatively simple autonomous behavior case we have two (autonomy) states: `LANE_FOLLOWING`, and `NORMAL_JOYSTICK_CONTROL` that can  be toggled using the `keyboard_control` GUI. For details, proceed to the [the next notebook about the finite state machine](./05_finite_state_machine.ipynb).
